In [1]:
import os
import csv
import pandas as pd
from datetime import datetime
from pyflink.table.expressions import col, lit
from pyflink.table.window import Slide, Tumble
from pyflink.table.udf import udf
from pyflink.table import (
    EnvironmentSettings,
    TableEnvironment,
    DataTypes
)

In [2]:
env_settings = (
    EnvironmentSettings.new_instance()
    .in_streaming_mode()
    .build()
)
t_env = TableEnvironment.create(env_settings)
conf = t_env.get_config().get_configuration()
conf.set_string("execution.target", "remote")
conf.set_string("rest.address", "jobmanager")
conf.set_string("rest.port", "8081")
conf.set_string("parallelism.default", "1")

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
t_env.execute_sql("DROP TABLE IF EXISTS aggtrades_source")
t_env.execute_sql("""
CREATE TABLE aggtrades_source (
  agg_trade_id BIGINT,
  price DOUBLE,
  quantity DOUBLE,
  first_trade_id BIGINT,
  last_trade_id BIGINT,
  ts_int BIGINT,
  is_buyer_maker BOOLEAN,
  is_best_match BOOLEAN,
  ts AS TO_TIMESTAMP_LTZ(ts_int / 1000, 3),
  WATERMARK FOR ts AS ts - INTERVAL '5' MINUTES
) with (
    'connector' = 'filesystem',
    'path' = '/workspace/ADAUSDT-aggTrades-2025-09-27.csv',
    'format' = 'csv'
)
""")

In [4]:
t_env.execute_sql("DROP TABLE IF EXISTS klines_sink")
t_env.execute_sql("""
create table klines_sink (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE
) with (
    'connector' = 'filesystem',
    'path' = '/workspace/output/klines/ADAUSDT/2025-09-27',
    'format' = 'csv'
)
""")

In [5]:
t_env.execute_sql("""
INSERT INTO klines_sink
SELECT
    window_start,
    window_end,
    ROUND(FIRST_VALUE(price), 4) AS open_price,
    ROUND(MAX(price), 4) AS high_price,
    ROUND(MIN(price), 4) AS low_price,
    ROUND(LAST_VALUE(price), 4) AS close_price,
    ROUND(SUM(quantity), 1) AS volume
FROM TABLE(
    TUMBLE(TABLE aggtrades_source, DESCRIPTOR(ts), INTERVAL '15' MINUTES)
)
GROUP BY window_start, window_end
""")

In [7]:
t_env.from_path("klines_sink").to_pandas().tail(20)

,window_start,window_end,open_price,high_price,low_price,close_price,volume
76,2025-09-27 19:00:00,2025-09-27 19:15:00,0.7806,0.7812,0.7801,0.7804,106570.2
77,2025-09-27 19:15:00,2025-09-27 19:30:00,0.7803,0.7823,0.7800,0.7821,95233.0
78,2025-09-27 19:30:00,2025-09-27 19:45:00,0.7822,0.7823,0.7802,0.7805,245898.9
79,2025-09-27 19:45:00,2025-09-27 20:00:00,0.7805,0.7819,0.7802,0.7815,79349.6
80,2025-09-27 20:00:00,2025-09-27 20:15:00,0.7815,0.7823,0.7811,0.7813,79122.1
81,2025-09-27 20:15:00,2025-09-27 20:30:00,0.7813,0.7813,0.7793,0.7798,846957.4
82,2025-09-27 20:30:00,2025-09-27 20:45:00,0.7798,0.7811,0.7793,0.7801,389823.6
83,2025-09-27 20:45:00,2025-09-27 21:00:00,0.7801,0.7801,0.7786,0.7797,453120.6
84,2025-09-27 21:00:00,2025-09-27 21:15:00,0.7797,0.7807,0.7789,0.7804,225027.5
85,2025-09-27 21:15:00,2025-09-27 21:30:00,0.7803,0.7810,0.7802,0.7810,97851.8
